# ESAT Workflow — GPU Accelerated

This notebook follows the official EPA ESAT workflow with GPU-accelerated training. Changes:
- `use_gpu=True, parallel=False` on BatchSA, Bootstrap, DISP, BS-DISP
- GPU batched multi-model training (`ls_nmf_batched`)
- Delta-optimised DISP binary search
- Backend metadata recorded


### Imports

In [1]:
import os, sys, json, warnings, logging
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
sys.path.insert(0, os.path.abspath(".."))
from esat.data.datahandler import DataHandler
from esat.model.sa import SA
from esat.model.batch_sa import BatchSA
from esat.data.analysis import ModelAnalysis, BatchAnalysis


### Dataset Paths

In [2]:
cwd = os.getcwd()
data_dir = os.path.join(cwd, "..", "data")
br_input_file = os.path.join(data_dir, "Dataset-BatonRouge-con.csv")
br_uncertainty_file = os.path.join(data_dir, "Dataset-BatonRouge-unc.csv")
br_output_path = os.path.join(data_dir, "output", "BatonRouge")
b_input_file = os.path.join(data_dir, "Dataset-Baltimore_con.txt")
b_uncertainty_file = os.path.join(data_dir, "Dataset-Baltimore_unc.txt")
b_output_path = os.path.join(data_dir, "output", "Baltimore")
sl_input_file = os.path.join(data_dir, "Dataset-StLouis-con.csv")
sl_uncertainty_file = os.path.join(data_dir, "Dataset-StLouis-unc.csv")
sl_output_path = os.path.join(data_dir, "output", "StLouis")


### Input Parameters

In [3]:
index_col = "Date"
factors = 6
method = "ls-nmf"
models = 20
init_method = "col_means"
init_norm = True
seed = 42
max_iterations = 20000
converge_delta = 0.01
converge_n = 50
verbose = True
use_gpu = True
parallel = False


### Dataset Selection

In [4]:
input_file = br_input_file
uncertainty_file = br_uncertainty_file
output_path = br_output_path


### Load Data

In [5]:
data_handler = DataHandler(input_path=input_file, uncertainty_path=uncertainty_file, index_col=index_col)
V, U = data_handler.get_data()
print("V=%s  (%d features, %d samples)" % (V.shape, len(data_handler.features), V.shape[0]))


V=(307, 41)  (41 features, 307 samples)


### Data Metrics

In [6]:
data_handler.metrics


,Category,S/N,Min,25th,50th,75th,Max
124-Trimethylbenzene,strong,5.445168,0.005000,0.820001,1.290001,1.865001,5.470003
224-Trimethylpentane,strong,5.666667,0.410000,1.580001,2.490002,3.865002,13.560008
234-Trimethylpentane,strong,5.537459,0.005000,0.530000,0.820001,1.300001,4.410003
23-Dimethylbutane,strong,5.500543,0.005000,0.640000,1.110001,2.285001,10.500007
23-Dimethylpentane,strong,5.463626,0.005000,0.340000,0.490000,0.780000,3.310002
2-Methylheptane,strong,5.039088,0.005000,0.215000,0.330000,0.535000,2.480002
3-Methylhexane,strong,5.648208,0.005000,0.655000,1.050001,1.510001,7.780005
3-Methylpentane,strong,5.611292,0.540000,1.720001,2.990002,5.945004,29.100018
Acetylene,strong,5.666667,0.380000,1.410001,1.990001,2.835002,8.070005
Benzene,strong,5.666667,0.590000,1.960001,2.770002,4.440003,9.330006


In [7]:
# data_handler.set_category(feature="Unidentified", category="bad")
data_handler.metrics


,Category,S/N,Min,25th,50th,75th,Max
124-Trimethylbenzene,strong,5.445168,0.005000,0.820001,1.290001,1.865001,5.470003
224-Trimethylpentane,strong,5.666667,0.410000,1.580001,2.490002,3.865002,13.560008
234-Trimethylpentane,strong,5.537459,0.005000,0.530000,0.820001,1.300001,4.410003
23-Dimethylbutane,strong,5.500543,0.005000,0.640000,1.110001,2.285001,10.500007
23-Dimethylpentane,strong,5.463626,0.005000,0.340000,0.490000,0.780000,3.310002
2-Methylheptane,strong,5.039088,0.005000,0.215000,0.330000,0.535000,2.480002
3-Methylhexane,strong,5.648208,0.005000,0.655000,1.050001,1.510001,7.780005
3-Methylpentane,strong,5.611292,0.540000,1.720001,2.990002,5.945004,29.100018
Acetylene,strong,5.666667,0.380000,1.410001,1.990001,2.835002,8.070005
Benzene,strong,5.666667,0.590000,1.960001,2.770002,4.440003,9.330006


In [8]:
data_handler.plot_data_uncertainty(feature_idx=0, include_menu=False)


In [9]:
data_handler.plot_feature_data(x_idx=0, y_idx=1)


In [10]:
data_handler.plot_feature_timeseries(feature_selection=[0])


In [11]:
# data_handler.plot_2d_histogram(x_col="124-Trimethylbenzene", y_col="224-Trimethylpentane")


### Train Model (GPU)

In [12]:
%%capture
sa_models = BatchSA(V=V, U=U, factors=factors, models=models, method=method, seed=seed,
                    max_iter=max_iterations, init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n,
                    verbose=verbose, use_gpu=use_gpu, parallel=parallel)
_ = sa_models.train()


[esat_rust] Metal device initialized successfully


In [13]:
best_model = sa_models.best_model
sa_model = sa_models.results[best_model]
print("Best model: %d, Q(true)=%.1f" % (best_model+1, sa_model.Qtrue))


Best model: 17, Q(true)=64616.6


### Save / Load (optional)

In [14]:
# batch_name = "test_batch_01"
# batch_output_dir = os.path.join(os.getcwd(), "test_batch")
# os.mkdir(batch_output_dir)
# header = data_handler.features
# sa_models.save(batch_name=batch_name, output_directory=batch_output_dir, pickle_model=True, pickle_batch=True, header=header)


In [15]:
# saved_batch_file = os.path.join(os.getcwd(), "test_batch", "test_batch_01.pkl")
# batch_sa2 = BatchSA.load(file_path=saved_batch_file)


### Post-Training Analysis

In [16]:
batch_analysis = BatchAnalysis(batch_sa=sa_models, data_handler=data_handler)
plot_test = batch_analysis.plot_loss(show=False)
plot_test


In [17]:
batch_analysis.plot_loss_distribution()


In [18]:
batch_analysis.plot_temporal_residuals(feature_idx=2)


In [19]:
batch_analysis.aggregated_output[0]


,124-Trimethylbenzene,224-Trimethylpentane,234-Trimethylpentane,23-Dimethylbutane,23-Dimethylpentane,2-Methylheptane,3-Methylhexane,3-Methylpentane,Acetylene,Benzene,...,O-Ethyltoluene,O-Xylene,Propane,Propylene,Styrene,Toluene,Trans-2-Butene,Trans-2-Pentene,Unidentified,TNMOC
0,0.184512,1.244582,0.379653,0.622446,0.284144,0.148353,0.705168,1.687358,1.053434,1.719147,...,0.043490,0.616470,8.687680,1.155071,0.006344,3.088368,0.010496,0.348974,6.446665,107.137138
1,2.428840,4.077993,1.543661,2.095303,0.834247,0.596980,1.202579,4.589378,2.908373,3.859955,...,0.736471,1.870334,14.736505,4.240337,1.130519,9.176787,0.369185,1.401220,25.717904,231.325717
2,1.353336,3.105354,1.039671,2.012180,0.648008,0.382346,1.176027,4.914517,2.110875,3.247630,...,0.372512,1.251990,16.151322,3.485749,0.560693,6.523628,0.309146,1.549927,25.080579,234.621531
3,5.205173,6.249464,2.223050,1.332309,0.989485,0.679815,0.901331,4.177775,5.568485,6.624315,...,1.533585,3.276769,17.880854,5.248813,1.827397,16.211806,0.259283,1.246385,62.118581,285.463509
4,1.397920,2.720126,0.875267,0.891619,0.510252,0.365848,0.838304,2.688790,2.402423,3.588934,...,0.443973,1.489025,13.782626,2.324397,0.552958,7.587694,0.009539,0.661668,9.227086,171.709308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,0.302810,2.196949,0.554277,3.074965,0.580090,0.317097,1.204388,7.400673,0.620474,2.217390,...,0.068692,0.429836,17.549136,2.788346,0.284045,3.490221,0.327249,3.113312,8.817008,253.417619
303,0.204440,1.120506,0.261071,1.290885,0.275190,0.133244,0.597772,3.314934,0.510766,1.320519,...,0.044409,0.303849,8.973082,1.180065,0.075986,2.150488,0.093992,1.382011,6.327609,124.498087
304,0.129159,0.491204,0.063170,0.156075,0.049241,0.018508,0.121835,0.954504,0.509666,1.007739,...,0.032071,0.241034,5.222628,0.618414,0.018092,1.502960,0.002893,0.201984,3.398815,56.357284
305,0.812346,1.436844,0.368046,0.501876,0.234623,0.058646,0.464564,1.942067,1.320871,1.884089,...,0.189255,0.647854,8.242635,1.091182,0.055622,3.529034,0.009302,0.670218,22.681847,114.424270


In [20]:
model_analysis = ModelAnalysis(datahandler=data_handler, model=sa_model, selected_model=best_model)


In [21]:
abs_threshold = 3.0
threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)


In [22]:
model_analysis.features_metrics()


,Feature,Input Mean,Input Var,Est Mean,Est Var,RMSE
0,124-Trimethylbenzene,1.4470,0.8667,1.1547,0.4850,0.8246
1,224-Trimethylpentane,3.0720,4.7755,2.6794,2.4659,1.3679
2,234-Trimethylpentane,1.0331,0.5468,0.8692,0.3339,0.5257
3,23-Dimethylbutane,1.8203,3.0425,1.3683,2.2522,1.5916
4,23-Dimethylpentane,0.6146,0.1861,0.5513,0.1347,0.2701
5,2-Methylheptane,0.3946,0.0888,0.3233,0.0708,0.2636
6,3-Methylhexane,1.2048,0.6763,1.0007,0.4245,0.6107
7,3-Methylpentane,4.8781,21.8641,3.4068,8.1883,4.4258
8,Acetylene,2.3288,1.7813,2.0843,1.2853,0.8496
9,Benzene,3.3562,3.5024,3.0388,2.2684,1.1898


In [23]:
model_analysis.calculate_statistics()
model_analysis.statistics


,Features,Category,r2,Intercept,Intercept SE,Slope,Slope SE,SE,SE Regression,Anderson Normal Residual,Anderson Statistic,Shapiro Normal Residuals,Shapiro PValue,KS Normal Residuals,KS PValue,KS Statistic
0,124-Trimethylbenzene,strong,0.598333,0.317437,0.046709,0.578633,0.027147,0.033701,0.441359,15.0,0.461426,Yes,1.103956e-01,Yes,8.920712e-01,0.032463
1,224-Trimethylpentane,strong,0.816320,0.684955,0.066482,0.649245,0.017634,0.058216,0.673009,15.0,0.221996,Yes,6.926874e-01,Yes,9.566740e-01,0.028629
2,234-Trimethylpentane,strong,0.731947,0.178530,0.029433,0.668553,0.023166,0.022073,0.299169,No,2.090344,No,1.781553e-04,Yes,1.434398e-01,0.064957
3,23-Dimethylbutane,strong,0.549750,0.207063,0.083341,0.637934,0.033057,0.067841,1.007012,No,1.181866,No,1.201227e-03,Yes,3.157259e-01,0.054235
4,23-Dimethylpentane,strong,0.772010,0.091947,0.017464,0.747441,0.023258,0.011776,0.175218,No,4.083753,No,1.925900e-08,No,3.872562e-02,0.079581
5,2-Methylheptane,strong,0.575684,0.056041,0.016467,0.677397,0.033300,0.011314,0.173348,No,1.562348,No,2.551971e-04,Yes,3.340349e-01,0.053380
6,3-Methylhexane,strong,0.712165,0.195256,0.035500,0.668549,0.024337,0.025298,0.349540,No,3.921630,No,5.821765e-10,No,6.759968e-03,0.095660
7,3-Methylpentane,strong,0.542567,1.207845,0.160146,0.450774,0.023700,0.183531,1.935357,No,1.422894,No,1.430768e-03,Yes,4.315883e-01,0.049270
8,Acetylene,strong,0.775823,0.341907,0.061813,0.748185,0.023029,0.036145,0.536773,15.0,0.221399,Yes,6.802330e-01,Yes,7.453241e-01,0.038244
9,Benzene,strong,0.776537,0.658662,0.083708,0.709186,0.021784,0.051147,0.711972,10.0,0.581265,Yes,2.174830e-01,Yes,6.621632e-01,0.041091


In [24]:
_ = model_analysis.plot_estimated_observed(feature_idx=0)


In [25]:
model_analysis.plot_estimated_timeseries(feature_idx=0)


In [26]:
model_analysis.plot_factor_profile(factor_idx=1)


In [27]:
model_analysis.plot_all_factors()


In [28]:
model_analysis.plot_all_factors_3d()


In [29]:
model_analysis.plot_factor_fingerprints()


In [30]:
model_analysis.plot_g_space(factor_1=2, factor_2=1)


In [31]:
model_analysis.plot_factor_contributions(feature_idx=1)


In [32]:
model_analysis.plot_factor_composition()


In [33]:
model_analysis.plot_factor_surface(factor_idx=None, feature_idx=1, percentage=True)


## Error Estimation — Displacement (GPU)

In [34]:
from esat.error.displacement import Displacement


In [35]:
disp = Displacement(sa=sa_model, feature_labels=data_handler.features,
                     model_selected=best_model, features=[0,1,2],
                     use_gpu=use_gpu, parallel=parallel)
print("DISP backend: %s" % ("gpu" if disp.use_gpu else "cpu"))


DISP backend: gpu


In [36]:
%%time
disp.run()


+ : Batch 1, Factor 1 - Features:   0%|          | 0/3 [00:00<?, ?it/s]

+ : Batch 1, Factor 1 - Features: 100%|██████████| 3/3 [00:00<00:00, 1556.14it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 2 - Features: 100%|██████████| 3/3 [00:00<00:00, 1853.97it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 3 - Features: 100%|██████████| 3/3 [00:00<00:00, 1716.16it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 4 - Features: 100%|██████████| 3/3 [00:00<00:00, 2080.51it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 5 - Features: 100%|██████████| 3/3 [00:00<00:00, 1930.78it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 6 - Features: 100%|██████████| 3/3 [00:00<00:00, 1830.51it/s]
[esat_rust] Metal device initialized successfully
- : Batch 1, Factor 1 - Features: 100%|██████████| 3/3 [00:00<00:00, 1434.60it/s]
[esat_rust] Metal device initialized successfully
- : Batch 1, Factor 2 - Features: 100%|██████████| 3/3 [00:00<00:00, 2001.42

CPU times: user 5.55 s, sys: 2.77 s, total: 8.31 s
Wall time: 18.7 s


In [37]:
disp.summary()


In [38]:
disp.plot_results(factor=3)


## Error Estimation — Bootstrap (GPU)

In [39]:
from esat.error.bootstrap import Bootstrap


In [40]:
model_selected = sa_models.best_model
nmf_model = sa_models.results[model_selected]
feature_labels = data_handler.features
bootstrap_n = 20
block_size = data_handler.optimal_block
threshold = 0.6
print("Optimal BS block size: %d" % data_handler.optimal_block)


Optimal BS block size: 4


In [41]:
bs = Bootstrap(sa=sa_model, feature_labels=feature_labels, model_selected=model_selected,
               bootstrap_n=bootstrap_n, block_size=block_size, threshold=threshold,
               seed=seed, use_gpu=use_gpu, parallel=parallel)


In [42]:
%%time
bs.run()


CPU times: user 499 ms, sys: 238 ms, total: 737 ms
Wall time: 2.03 s


[esat_rust] Metal device initialized successfully


In [43]:
bs.summary()


In [44]:
bs.q_results


,Q(robust)
0,53726.689088
1,50287.148232
2,54804.059133
3,53185.306735
4,53945.720392
5,52216.417524
6,54918.566843
7,54217.642196
8,54777.749419
9,54034.414776


In [45]:
# bs.plot_results(factor=1)


## Error Estimation — BS-DISP (GPU)

In [46]:
from esat.error.bs_disp import BSDISP


In [47]:
model_selected = sa_models.best_model
sa_model = sa_models.results[model_selected]
bootstrap_n = 10
block_size = data_handler.optimal_block
threshold = 0.6
threshold_dQ = 0.1
max_search = 50
features = [0, 1, 2]


In [48]:
bsdisp = BSDISP(sa=sa_model, feature_labels=feature_labels, model_selected=model_selected,
                  bootstrap_n=bootstrap_n, block_size=block_size, threshold=threshold,
                  max_search=max_search, threshold_dQ=threshold_dQ, features=features,
                  seed=seed, use_gpu=use_gpu)


In [49]:
%%capture
bsdisp.run(parallel=parallel)


[esat_rust] Metal device initialized successfully


/Users/joallee/Project/esat-gpu-study/esat/metrics.py:15: RuntimeWarning: divide by zero encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/metrics.py:15: RuntimeWarning: overflow encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/metrics.py:15: RuntimeWarning: invalid value encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/metrics.py:27: RuntimeWarning: divide by zero encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/metrics.py:27: RuntimeWarning: overflow encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/metrics.py:27: RuntimeWarning: invalid value encountered in matmul
  _wh = np.matmul(W, H)
/Users/joallee/Project/esat-gpu-study/esat/model/sa.py:467: RuntimeWarning: divide by zero encountered in matmul
  self.WH = np.matmul(self.W, self.H)
/Users/joallee/Project/esat-gpu-study/esat/model/sa.p

In [50]:
bsdisp.summary()


In [51]:
bsdisp.plot_results(factor=1)


### Combined Error Plot

In [52]:
from esat.error.error import Error


In [53]:
error = Error(bs=bs, disp=disp, bsdisp=bsdisp)


In [54]:
error.plot_summary(factor=1)


## Constrained Model

In [55]:
from esat.rotational.constrained import ConstrainedModel
cm = ConstrainedModel(base_model=sa_model, data_handler=data_handler, softness=1.0)


In [56]:
cm.add_constraint(constraint_type="define limits", index=(2,10), target="feature", min_value=0.1, max_value=0.9)


True

In [57]:
cm.list_constraints()


In [60]:
cm.remove_constraint(constraint_label="factor:0|feature:3")


KeyError: 'factor:0|feature:3'

In [61]:
expression1 = "(0.66*[factor:1|feature:2])-(4.2*[factor:2|feature:4])=0,250"
expression2 = "(0.35*[factor:0|feature:3])-(2.0*[factor:1|feature:3])-(3.7*[factor:3|feature:4])=0,250"
cm.add_expression(expression1)
cm.add_expression(expression2)


True

In [62]:
cm.list_expressions()


In [63]:
cm.train(max_iterations=20000)


Q(Robust): 58001.516, Q(Main): 68941.067, Q(aux): 559.86, dQ: 0.0121:   2%|▏         | 421/20000 [00:02<02:00, 162.48it/s]  


In [64]:
cm.display_results()
cm.plot_Q(Qtype="Aux")


In [65]:
cm.evaluate_constraints()


In [66]:
cm.evaluate_expressions()


In [67]:
cm.plot_profile_contributions(factor_idx=1)


In [68]:
cm.plot_factor_fingerprints()


In [69]:
cm.plot_g_space(factor_idx1=1, factor_idx2=2, show_base=True, show_delta=True)


In [70]:
cm.plot_factor_contributions(feature_idx=4)


In [71]:
cm_model = cm.constrained_model


## Error Estimation on Constrained Model (GPU)

In [72]:
cm_disp = Displacement(sa=cm_model, feature_labels=data_handler.features,
                          model_selected="constrained", features=[0,1,2,3,4],
                          use_gpu=use_gpu, parallel=parallel)


In [73]:
%%time
cm_disp.run()


+ : Batch 1, Factor 1 - Features: 100%|██████████| 5/5 [00:00<00:00, 2104.31it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 2 - Features: 100%|██████████| 5/5 [00:00<00:00, 2341.88it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 3 - Features: 100%|██████████| 5/5 [00:00<00:00, 2117.69it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 4 - Features: 100%|██████████| 5/5 [00:00<00:00, 2232.20it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 5 - Features: 100%|██████████| 5/5 [00:00<00:00, 2281.99it/s]
[esat_rust] Metal device initialized successfully
+ : Batch 1, Factor 6 - Features: 100%|██████████| 5/5 [00:00<00:00, 1790.30it/s]
[esat_rust] Metal device initialized successfully
- : Batch 1, Factor 4 - Features: 100%|██████████| 5/5 [00:00<00:00, 1533.23it/s]
[esat_rust] Metal device initialized successfully
- : Batch 1, Factor 5 - Features: 100%|██████████| 5/5 [00:00<00:00, 1458.48

CPU times: user 5.67 s, sys: 2.83 s, total: 8.5 s
Wall time: 24.4 s


In [74]:
cm_disp.summary()


In [75]:
cm_disp.plot_results(factor=1)


In [76]:
cm_bs = Bootstrap(sa=cm_model, feature_labels=feature_labels, model_selected="constrained",
                   bootstrap_n=20, block_size=data_handler.optimal_block, threshold=0.6,
                   seed=seed, use_gpu=use_gpu, parallel=parallel)


In [77]:
%%time
cm_bs.run()


CPU times: user 494 ms, sys: 234 ms, total: 729 ms
Wall time: 2.07 s


[esat_rust] Metal device initialized successfully


In [78]:
cm_bs.summary()


In [79]:
cm_bs.plot_results(factor=1)
